# ASC Phase 3 — RoBERTa Training

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas numpy tqdm

## 1. Configuration

In [2]:
from pathlib import Path

SEED = 42

TRAIN_PATH = "/content/drive/MyDrive/absa_self_train_phase3/train.csv"
TEST_PATH = "/content/drive/MyDrive/absa_self_train_phase3/gold_test.csv"

BASE_MODEL_NAME = "roberta-base"

OUTPUT_DIR = "/content/asc_phase3_roberta_tmp"
SAVE_DIR = "/content/drive/MyDrive/absa_self_train_phase3/model"

MAX_LENGTH = 192
# VALID_SIZE = 0.05

NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

EVAL_STEPS = 1000
LOGGING_STEPS = 1000
EARLY_STOPPING_PATIENCE = 3

SPECIAL_TOKENS = ["[ASP]", "[/ASP]"]

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
!rm -rf /content/asc_phase2_roberta_tmp

## 2. Imports

In [3]:
import ast
import re
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

LABEL_ID_TO_NAME = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

LABEL_NAME_TO_ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

device: cuda


## 3. Load data

In [4]:
train_raw = pd.read_csv(TRAIN_PATH, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, low_memory=False)

print("train raw:", train_raw.shape)
print("gold test raw:", test_raw.shape)

display(train_raw.head())
display(test_raw.head())

train raw: (2000253, 7)
gold test raw: (782, 7)


,parent_asin,sentence_id,sentence_text,rating,category_name,aspect,sentiment
0,B09L9YLCNG,1,i would have rather seen this book go on about...,3.0,kindle_store,book,"[0.014366569928824902, 0.005626029334962368, 0..."
1,B07KB8DB1S,1,omg this book was [GENERIC_NOUN] of those book...,5.0,kindle_store,books,"[0.9911634922027588, 4.9421461881138384e-05, 0..."
2,B08HXJYQZW,2,firstly i enjoy books that illuminate the pers...,5.0,kindle_store,reader,"[0.005310497712343931, 0.0032304299529641867, ..."
3,B013GBA5TA,4,good body design and construction,5.0,electronics_p2,construction,"[0.002743832068517804, 0.003060966031625867, 0..."
4,B0BJJ6JLV8,3,this cable is so nice for when i need both con...,5.0,electronics_p2,cable,"[0.003357902867719531, 0.002633075462654233, 0..."


,parent_asin,sentence_id,sentence_text,rating,aspect,sentiment,category_name
0,B089QL57J5,1,perfect stand for a novation launchpad x i use...,5.0,stand,"[0.0, 0.0, 1.0]",electronics_p2
1,B000HCZ8EO,686644,this transaction was smooth,5.0,transaction,"[0.0, 0.0, 1.0]",software
2,B0036D5XH8,2,the handset is lightweight and very comfortabl...,5.0,handset,"[0.0, 0.0, 1.0]",office_products
3,B0036D5XH8,2,the handset is lightweight and very comfortabl...,5.0,handset,"[0.0, 0.0, 1.0]",office_products
4,B0036D5XH8,2,the handset is lightweight and very comfortabl...,5.0,clips,"[0.0, 0.0, 1.0]",office_products


## 4. Preprocessing

In [5]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_sentiment_label(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, (float, np.floating)) and not pd.isna(value):
        return int(value)

    if isinstance(value, list):
        return int(np.argmax(value))

    text = str(value).strip()

    if text in {"0", "1", "2"}:
        return int(text)

    parsed = ast.literal_eval(text)

    if isinstance(parsed, list):
        return int(np.argmax(parsed))

    return int(parsed)


def mark_aspect(sentence, aspect):
    sentence = clean_text(sentence)
    aspect = clean_text(aspect)

    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    return pattern.sub(f"[ASP] {aspect} [/ASP]", sentence, count=1)

In [6]:
def prepare_asc_df(df):
    data = df.copy()

    data["parent_asin"] = data["parent_asin"].astype(str)
    data["sentence_id"] = data["sentence_id"].astype(str)
    data["sentence_text"] = data["sentence_text"].apply(clean_text)
    data["aspect"] = data["aspect"].apply(clean_text)
    data["label"] = data["sentiment"].apply(parse_sentiment_label)
    data["category_name"] = data["category_name"].astype(str)

    data = data[
        (data["sentence_text"].str.len() > 0) &
        (data["aspect"].str.len() > 0) &
        (data["label"].isin([0, 1, 2]))
    ].copy()

    data["input_text"] = [
        mark_aspect(sentence, aspect)
        for sentence, aspect in zip(data["sentence_text"], data["aspect"])
    ]

    return data.reset_index(drop=True)


train_df = prepare_asc_df(train_raw)
test_df = prepare_asc_df(test_raw)

print("train:", train_df.shape)
print("gold test:", test_df.shape)

display(train_df[["sentence_text", "aspect", "sentiment", "label", "input_text", "category_name"]].head())

train: (2000252, 9)
gold test: (782, 9)


,sentence_text,aspect,sentiment,label,input_text,category_name
0,i would have rather seen this book go on about...,book,"[0.014366569928824902, 0.005626029334962368, 0...",2,i would have rather seen this [ASP] book [/ASP...,kindle_store
1,omg this book was [GENERIC_NOUN] of those book...,books,"[0.9911634922027588, 4.9421461881138384e-05, 0...",0,omg this book was [GENERIC_NOUN] of those [ASP...,kindle_store
2,firstly i enjoy books that illuminate the pers...,reader,"[0.005310497712343931, 0.0032304299529641867, ...",2,firstly i enjoy books that illuminate the pers...,kindle_store
3,good body design and construction,construction,"[0.002743832068517804, 0.003060966031625867, 0...",2,good body design and [ASP] construction [/ASP],electronics_p2
4,this cable is so nice for when i need both con...,cable,"[0.003357902867719531, 0.002633075462654233, 0...",2,this [ASP] cable [/ASP] is so nice for when i ...,electronics_p2


## 5. Label distribution

In [7]:
def label_distribution(df, name):
    out = (
        df["label"]
        .map(LABEL_ID_TO_NAME)
        .value_counts()
        .rename_axis("label")
        .reset_index(name="count")
    )
    out["ratio"] = out["count"] / out["count"].sum()
    print(name)
    display(out)


label_distribution(train_df, "train")
label_distribution(test_df, "gold test")

print("train by category")
display(pd.crosstab(train_df["category_name"], train_df["label"].map(LABEL_ID_TO_NAME)))

print("gold test by category")
display(pd.crosstab(test_df["category_name"], test_df["label"].map(LABEL_ID_TO_NAME)))

train


,label,count,ratio
0,negative,806427,0.403163
1,positive,804723,0.402311
2,neutral,389102,0.194526


gold test


,label,count,ratio
0,positive,496,0.634271
1,negative,256,0.327366
2,neutral,30,0.038363


train by category


label,negative,neutral,positive
category_name,,,
Software,199987,0,197890
electronics_p1,137,21,244
electronics_p2,116376,167691,116574
kindle_store,149625,100948,149363
office_products,340084,120401,340280
software,218,41,372


gold test by category


label,negative,neutral,positive
category_name,,,
electronics_p1,35,5,65
electronics_p2,47,3,109
kindle_store,30,2,115
office_products,79,9,108
software,65,11,99


## 6. Train/validation split

In [8]:
# train_part, valid_part = train_test_split(
#     train_df,
#     test_size=VALID_SIZE,
#     random_state=SEED,
#     stratify=train_df["label"],
# )

# train_part = train_part.reset_index(drop=True)
# valid_part = valid_part.reset_index(drop=True)

train_part = train_df.copy()
valid_part = test_df.copy()

print("train part:", train_part.shape)
print("valid part:", valid_part.shape)

label_distribution(train_part, "train part")
label_distribution(valid_part, "valid part")

train part: (2000252, 9)
valid part: (782, 9)
train part


,label,count,ratio
0,negative,806427,0.403163
1,positive,804723,0.402311
2,neutral,389102,0.194526


valid part


,label,count,ratio
0,positive,496,0.634271
1,negative,256,0.327366
2,neutral,30,0.038363


## 7. Tokenization

In [9]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})


def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[["input_text", "label"]],
        preserve_index=False,
    )


def tokenize_batch(batch):
    return tokenizer(
        batch["input_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


train_ds = to_hf_dataset(train_part).map(tokenize_batch, batched=True, remove_columns=["input_text"])
valid_ds = to_hf_dataset(valid_part).map(tokenize_batch, batched=True, remove_columns=["input_text"])
test_ds = to_hf_dataset(test_df).map(tokenize_batch, batched=True, remove_columns=["input_text"])

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_ds)
print(valid_ds)
print(test_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000252 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 2000252
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 782
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 782
})


## 8. Metrics

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
    }


def report_df(y_true, y_pred):
    return pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            target_names=["negative", "neutral", "positive"],
            output_dict=True,
            zero_division=0,
        )
    ).T


def confusion_df(y_true, y_pred):
    return pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
        index=["gold_negative", "gold_neutral", "gold_positive"],
        columns=["pred_negative", "pred_neutral", "pred_positive"],
    )

## 9. Model

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=3,
    id2label=LABEL_ID_TO_NAME,
    label2id=LABEL_NAME_TO_ID,
)

model.resize_token_embeddings(len(tokenizer))
model.to(device)

label_counts = train_part["label"].value_counts().reindex([0, 1, 2], fill_value=0).to_numpy()
freq = label_counts / label_counts.sum()

class_weights = 1.0 / np.sqrt(np.maximum(freq, 1e-12))
class_weights = class_weights / class_weights.mean()
class_weights = np.clip(class_weights, 0.25, 5.0)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("label counts:", dict(zip(["negative", "neutral", "positive"], label_counts.tolist())))
print("class weights:", dict(zip(["negative", "neutral", "positive"], class_weights.tolist())))


class WeightedClassificationTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights = self.class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and

label counts: {'negative': 806427, 'neutral': 389102, 'positive': 804723}
class weights: {'negative': 0.8719186782836914, 'neutral': 1.2552399635314941, 'positive': 0.8728412985801697}


## 10. Training

In [12]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=SEED,
    data_seed=SEED,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    fp16=torch.cuda.is_available(),
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedClassificationTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
        )
    ],
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall,Weighted Precision,Weighted Recall
1000,1.040265,0.566478,0.827366,0.644975,0.828316,0.648585,0.652165,0.840011,0.827366
2000,0.248227,0.613002,0.875959,0.683431,0.870321,0.743222,0.669153,0.872417,0.875959
3000,0.161003,0.693634,0.883632,0.691989,0.876387,0.782402,0.668775,0.877346,0.883632
4000,0.152459,0.757595,0.882353,0.689420,0.875362,0.764571,0.668103,0.874984,0.882353
5000,0.137984,0.833964,0.881074,0.692900,0.874149,0.800654,0.670581,0.878546,0.881074
6000,0.136034,0.849490,0.884910,0.695919,0.877785,0.803831,0.673227,0.881755,0.884910
7000,0.142715,0.854476,0.878517,0.683771,0.872545,0.734577,0.668607,0.871724,0.878517
8000,0.140809,0.841456,0.873402,0.686072,0.865360,0.799029,0.657098,0.867377,0.873402
9000,0.127357,0.910666,0.878517,0.684709,0.872033,0.746841,0.666087,0.871124,0.878517


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=9000, training_loss=0.2540947028266059, metrics={'train_runtime': 1877.0509, 'train_samples_per_second': 3196.906, 'train_steps_per_second': 199.807, 'total_flos': 1.4208124557312e+16, 'train_loss': 0.2540947028266059, 'epoch': 0.07199078517949703})

## 11. Gold-test benchmark

In [13]:
pred_output = trainer.predict(test_ds)

logits = pred_output.predictions
y_pred = np.argmax(logits, axis=-1)
y_true = test_df["label"].to_numpy()

test_metrics = compute_metrics((logits, y_true))

print("Gold-test metrics")
display(pd.DataFrame([test_metrics]))

print("Classification report")
display(report_df(y_true, y_pred))

print("Confusion matrix")
display(confusion_df(y_true, y_pred))

test_pred_df = test_df.copy()
test_pred_df["pred_label"] = y_pred
test_pred_df["pred_name"] = test_pred_df["pred_label"].map(LABEL_ID_TO_NAME)
test_pred_df["gold_name"] = test_pred_df["label"].map(LABEL_ID_TO_NAME)

display(test_pred_df[["sentence_text", "aspect", "gold_name", "pred_name", "category_name"]].head(20))

Gold-test metrics


,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,weighted_precision,weighted_recall
0,0.88491,0.695919,0.877785,0.803831,0.673227,0.881755,0.88491


Classification report


,precision,recall,f1-score,support
negative,0.816254,0.902344,0.857143,256.00000
neutral,0.666667,0.200000,0.307692,30.00000
positive,0.928571,0.917339,0.922921,496.00000
accuracy,0.884910,0.884910,0.884910,0.88491
macro avg,0.803831,0.673227,0.695919,782.00000
weighted avg,0.881755,0.884910,0.877785,782.00000


Confusion matrix


,pred_negative,pred_neutral,pred_positive
gold_negative,231,1,24
gold_neutral,13,6,11
gold_positive,39,2,455


,sentence_text,aspect,gold_name,pred_name,category_name
0,perfect stand for a novation launchpad x i use...,stand,positive,positive,electronics_p2
1,this transaction was smooth,transaction,positive,positive,software
2,the handset is lightweight and very comfortabl...,handset,positive,positive,office_products
3,the handset is lightweight and very comfortabl...,handset,positive,positive,office_products
4,the handset is lightweight and very comfortabl...,clips,positive,positive,office_products
5,keep your electrical tape handy this cord work...,cord,positive,positive,electronics_p1
6,keep your electrical tape handy this cord work...,cord,negative,positive,electronics_p1
7,retractable makes convenient to put in pencil ...,retractable,positive,positive,office_products
8,this [GENERIC_NOUN] works great for cox intern...,connection,positive,positive,electronics_p1
9,good app i will say this app is pretty good bu...,app,positive,positive,software


## 12. Save outputs

In [14]:
best_model_dir = Path(SAVE_DIR) / "best_macro_f1"
report_dir = Path(SAVE_DIR) / "reports"

best_model_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(best_model_dir))
tokenizer.save_pretrained(str(best_model_dir))

metrics_path = report_dir / "gold_test_metrics.csv"
report_path = report_dir / "gold_test_classification_report.csv"
confusion_path = report_dir / "gold_test_confusion_matrix.csv"
pred_path = report_dir / "gold_test_predictions.csv"
text_report_path = report_dir / "gold_test_report.txt"

pd.DataFrame([test_metrics]).to_csv(metrics_path, index=False)
report_df(y_true, y_pred).to_csv(report_path)
confusion_df(y_true, y_pred).to_csv(confusion_path)
test_pred_df.to_csv(pred_path, index=False)

lines = [
    "ASC phase 3 benchmark",
    f"seed: {SEED}",
    f"base_model: {BASE_MODEL_NAME}",
    f"input_format: aspect_marker",
    f"train_rows: {len(train_df):,}",
    f"valid_rows: {len(valid_part):,}",
    f"gold_test_rows: {len(test_df):,}",
    "",
]

for key, value in test_metrics.items():
    lines.append(f"{key}: {value:.6f}")

text_report_path.write_text("\n".join(lines), encoding="utf-8")

print("saved model:", best_model_dir)
print("saved reports:", report_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved model: /content/drive/MyDrive/absa_self_train_phase3/model/best_macro_f1
saved reports: /content/drive/MyDrive/absa_self_train_phase3/model/reports


## 13. Reload check

In [15]:
reloaded_tokenizer = AutoTokenizer.from_pretrained(best_model_dir, use_fast=True)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir).to(device)
reloaded_model.eval()

sample = test_df.sample(min(5, len(test_df)), random_state=SEED)

enc = reloaded_tokenizer(
    sample["input_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    probs = torch.softmax(reloaded_model(**enc).logits, dim=-1).cpu().numpy()

sample_out = sample[["sentence_text", "aspect", "label", "category_name"]].copy()
sample_out["pred_label"] = probs.argmax(axis=-1)
sample_out["confidence"] = probs.max(axis=-1)

display(sample_out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,sentence_text,aspect,label,category_name,pred_label,confidence
596,j videoid26037f6e1218ea624523a34f23c2448b [GEN...,folders,0,office_products,0,0.999584
588,level 11 really liked this game until i got to...,letters,0,software,0,0.999133
208,this game is so addictive i have a hard [GENER...,game,2,software,2,0.999711
291,thoroughly enjoyed this intelligent and quietl...,mystery,2,kindle_store,2,0.999751
174,great software bad update policy i have owned ...,update policy,0,software,0,0.998212
